<a href="https://colab.research.google.com/github/MouslimRaza/smart-finance-assistant/blob/main/Starter_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio pandas hands-on-ai --quiet

In [ ]:
import pandas as pd
import numpy as np
import os, io, warnings
warnings.filterwarnings('ignore')

os.environ['HANDS_ON_AI_SERVER']  = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL']   = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = 'isys2001-assignment-key'

from hands_on_ai.chat import get_response
print("✅ Setup complete!")

In [ ]:
step1 = """
STEP 1: UNDERSTAND THE PROBLEM
===============================
Many young people who enjoy gaming, fitness, and entertainment
have no clear picture of where their money goes each month.
Subscriptions, game purchases, gym fees, cinema outings, and
café visits add up quickly without feeling significant in the moment.

The Smart Finance Assistant solves this by:
  1. Accepting a CSV file of personal transactions
  2. Automatically cleaning and categorising the data
  3. Showing a clear spending breakdown by category
  4. Offering a chatbot (FinBot) that answers finance questions
  5. Retrieving relevant saving tips from a knowledge base (RAG)
  6. Calculating how long it takes to reach a savings goal

TARGET USER:
  A young person who games on Xbox, goes to the gym, watches
  movies at MCine, and enjoys tea at Maison de Thé.
  Someone who wants quick spending insights without manually
  going through bank statements.

FINANCE PROBLEM BEING SOLVED:
  Lack of spending awareness — small daily purchases like tea at
  Maison de Thé and impulse Xbox game buys feel harmless but
  accumulate into hundreds of dollars of untracked spending per month.
"""
print(step1)

In [ ]:
step2 = """
STEP 2: IDENTIFY INPUTS AND OUTPUTS
=====================================

COMPONENT 1 — CSV Analysis
  Input  : CSV file with columns: Date, Amount, Category, Description
            Amounts may include $ signs and negatives (refunds)
  Output : Spending totals per category, percentages, recommendations

  Example Input Row:
    2026-03-05, $69.99, Gaming, Xbox Game Pass Ultimate 3 Months

  Example Output:
    Gaming      $312.45  (38.2%)  ████████
    Gym         $145.00  (17.8%)  ███
    Cinema      $ 86.00  (10.5%)  ██
    Tea & Café  $ 72.50   (8.9%)  █

COMPONENT 2 — Chatbot (FinBot)
  Input  : User question (e.g. "Am I spending too much on Xbox games?")
           Optional: spending summary from CSV as context
  Output : Plain-English personalised finance advice from FinBot

COMPONENT 3 — RAG (Finance Tips)
  Input  : User question (e.g. "How do I cut my gaming spend?")
  Output : Answer retrieved from the financial tips knowledge base

COMPONENT 4 — Savings Calculator
  Input  : Monthly income ($)
            Monthly expenses ($)
            Savings goal amount ($)
  Output : Months to reach goal, weekly/daily savings, savings rate %

COMPONENT 5 — Gradio UI
  Input  : All of the above via a browser interface
  Output : Formatted results displayed in the browser app
"""
print(step2)

In [ ]:
step3 = """
STEP 3: WORK THE PROBLEM BY HAND
==================================

--- Example 1: Cleaning an Amount ---

Raw value from CSV  :  "$69.99"
Step 1 — Remove $   :  "69.99"
Step 2 — To float   :  69.99
Result              :  69.99  ✓

Raw value from CSV  :  "-$15.00"
Step 1 — Remove $   :  "-15.00"
Step 2 — To float   :  -15.00
Result              :  -15.00  (this is a refund)  ✓

--- Example 2: Spending by Category ---

Transactions:
  2026-03-05  $69.99   Gaming      Xbox Game Pass Ultimate
  2026-03-08  $60.00   Gym         Monthly Gym Membership
  2026-03-12  $17.00   Cinema      MCine Moka - 2 Tickets Avengers
  2026-03-14  $8.50    Tea & Café  Maison de Thé - Green Tea Moka
  2026-03-20  -$15.00  Refund      Xbox Store - Refund Cancelled DLC

Positive spending only (ignore refunds):
  Gaming     = $69.99
  Gym        = $60.00
  Cinema     = $17.00
  Tea & Café = $8.50
  TOTAL      = $155.49

Percentages:
  Gaming     = 69.99 / 155.49 * 100 = 45.0%
  Gym        = 60.00 / 155.49 * 100 = 38.6%
  Cinema     = 17.00 / 155.49 * 100 = 10.9%
  Tea & Café =  8.50 / 155.49 * 100 =  5.5%
  CHECK: 45.0 + 38.6 + 10.9 + 5.5 = 100.0%  ✓

Refunds total = $15.00
Net spent     = $155.49 - $15.00 = $140.49

--- Example 3: Savings Calculator ---

Monthly income   = $2,500
Monthly expenses = $1,800
Monthly savings  = $2,500 - $1,800 = $700

Savings goal = $5,000 (e.g. new Xbox Series X + accessories + games)
Months needed = $5,000 / $700 = 7.14 months ≈ 8 months

Weekly savings = $700 / 4.33 = $161.66 per week
Savings rate   = $700 / $2,500 * 100 = 28.0%  (above 20% ✓)
"""
print(step3)

In [ ]:
step4 = """
STEP 4: PSEUDOCODE
===================

--- load_and_clean(file) ---
  READ csv file into a DataFrame
  CHECK that columns Date, Amount, Category, Description exist
    IF missing → raise error with helpful message
  FOR each row in Amount column:
    REMOVE dollar signs, commas, spaces using regex
    CONVERT to float (set invalid values to NaN)
  DROP rows where Amount could not be converted
  FILL missing Category with "Uncategorised"
  RETURN cleaned DataFrame

--- analyse_spending(df) ---
  SPLIT df into:
    positive_spending = rows where Amount > 0
    refunds           = rows where Amount < 0
  total_spent   = SUM of positive_spending amounts
  total_refunds = ABS(SUM of refund amounts)
  net_spent     = total_spent - total_refunds
  GROUP positive_spending by Category:
    calculate: sum, average, count per category
  CALCULATE percentage = category_total / total_spent * 100
  SORT categories by total descending
  RETURN dict with all the above values

--- make_recommendations(analysis) ---
  FOR each category in analysis:
    IF Gaming AND total > $50:
      ADD tip about 48-hour wishlist rule for Xbox purchases
    IF Gym AND total > $80:
      ADD tip about off-peak memberships and block PT sessions
    IF Cinema AND total > $40:
      ADD tip about MCine loyalty card and off-peak days
    IF Tea & Café AND total > $30:
      ADD tip about brewing tea at home instead of Maison de Thé daily
  IF no tips triggered:
    ADD generic positive message
  RETURN all tips as a formatted string

--- savings_calculator(income, expenses, goal) ---
  VALIDATE: income > 0, expenses >= 0, goal > 0
  monthly_savings = income - expenses
  IF monthly_savings <= 0:
    RETURN error message showing the shortfall amount
  months_needed = goal / monthly_savings
  weekly_saving = monthly_savings / 4.33
  savings_rate  = monthly_savings / income * 100
  RETURN formatted report with all calculations

--- chat_finbot(user_message, spending_context) ---
  BUILD system prompt with FinBot personality (Xbox/gym/MCine/Maison de Thé)
  IF spending_context available:
    APPEND context to system prompt
  CALL get_response(user_message, system=system_prompt)
  RETURN response text

--- rag_lookup(question) ---
  TRY hands_on_ai.rag.get_answer(question, documents)
  IF rag module not available:
    BUILD prompt: "Answer ONLY from this document: [tips]"
    CALL get_response(prompt)
  RETURN answer

--- Gradio UI ---
  CREATE app with 4 tabs:
    Tab 1: File upload → load_and_clean + analyse_spending + full_report
    Tab 2: Chat input + optional CSV → chat_finbot with spending context
    Tab 3: Text question → rag_lookup answer
    Tab 4: Income, expenses, goal → savings_calculator result
  LAUNCH app with share=True for public Colab link
"""
print(step4)

In [ ]:
SAMPLE_CSV = """Date,Amount,Category,Description
2026-03-01,$60.00,Gym,Monthly Gym Membership - March
2026-03-02,$8.50,Tea & Café,Maison de Thé - Green Tea Moka
2026-03-03,$17.00,Cinema,MCine Moka - 2 Tickets Avengers
2026-03-05,$69.99,Gaming,Xbox Game Pass Ultimate 3 Months
2026-03-07,$8.50,Tea & Café,Maison de Thé - Earl Grey & Pastry
2026-03-09,$24.99,Gaming,Xbox Store - EA FC 25 DLC Pack
2026-03-10,$12.00,Gaming,Xbox Store - Minecraft Marketplace Bundle
2026-03-12,$9.00,Cinema,MCine Moka - Solo Ticket Minecraft Movie
2026-03-14,$7.50,Tea & Café,Maison de Thé - Chai Latte Moka
2026-03-16,$35.00,Gaming,Xbox Store - Forza Horizon 5 Premium Add-Ons
2026-03-18,$8.50,Tea & Café,Maison de Thé - Matcha & Croissant
2026-03-20,-$15.00,Refund,Xbox Store - Refund Cancelled DLC
2026-03-21,$25.00,Gym,Personal Training Session
2026-03-23,$59.99,Gaming,Xbox Store - Call of Duty Black Ops 7
2026-03-25,$8.00,Tea & Café,Maison de Thé - Oolong Tea Moka
2026-03-27,$17.00,Cinema,MCine Moka - 2 Tickets Thunderbolts
2026-03-29,$14.99,Gaming,Xbox Store - Xbox Avatar Items Bundle
2026-03-31,$7.50,Tea & Café,Maison de Thé - Rooibos Tea & Cake
2026-04-01,$60.00,Gym,Monthly Gym Membership - April
2026-04-02,$49.99,Gaming,Xbox Store - Halo Infinite Season Pass
2026-04-04,$8.50,Tea & Café,Maison de Thé - Green Tea Moka
2026-04-05,$9.00,Cinema,MCine Moka - Solo Ticket Sinners
2026-04-07,$19.99,Gaming,Xbox Store - Rocket League Credits Pack
2026-04-09,$25.00,Gym,Personal Training Session
2026-04-10,$8.00,Tea & Café,Maison de Thé - Jasmine Tea Moka
2026-04-12,$17.00,Cinema,MCine Moka - 2 Tickets Mission Impossible
2026-04-14,$7.50,Tea & Café,Maison de Thé - Peppermint & Biscuit
2026-04-15,$39.99,Gaming,Xbox Store - Diablo IV Expansion Pack
2026-04-17,$8.50,Tea & Café,Maison de Thé - Earl Grey Moka
2026-04-18,-$9.99,Refund,Xbox Store - Refund Avatar Items
2026-04-19,$9.00,Cinema,MCine Moka - Solo Ticket Lilo and Stitch
2026-04-21,$12.00,Gaming,Xbox Store - Gears 6 Starter Pack
2026-04-23,$7.50,Tea & Café,Maison de Thé - Matcha Latte Moka
2026-04-25,$17.00,Cinema,MCine Moka - 2 Tickets Final Destination
2026-04-27,$8.50,Tea & Café,Maison de Thé - Chai & Croissant
2026-04-28,$25.00,Gym,Personal Training Session
2026-04-30,$29.99,Gaming,Xbox Store - Starfield Story DLC
2026-05-01,$60.00,Gym,Monthly Gym Membership - May
2026-05-02,$8.50,Tea & Café,Maison de Thé - Green Tea Moka
2026-05-04,$17.00,Cinema,MCine Moka - 2 Tickets Ballerina
2026-05-05,$24.99,Gaming,Xbox Game Pass Ultimate Monthly Renewal
2026-05-07,$9.00,Cinema,MCine Moka - Solo Ticket Karate Kid
2026-05-08,$7.50,Tea & Café,Maison de Thé - Oolong & Madeleine
2026-05-10,$34.99,Gaming,Xbox Store - Fable Deluxe Edition Pre-Order
2026-05-12,$8.50,Tea & Café,Maison de Thé - Rooibos Tea Moka
2026-05-14,$25.00,Gym,Personal Training Session
2026-05-15,$17.00,Cinema,MCine Moka - 2 Tickets Jurassic World
2026-05-17,$9.99,Gaming,Xbox Store - Fortnite V-Bucks Pack
2026-05-18,$7.50,Tea & Café,Maison de Thé - Jasmine & Biscuit
2026-05-20,$9.00,Cinema,MCine Moka - Solo Ticket 28 Years Later
2026-05-22,$8.50,Tea & Café,Maison de Thé - Matcha Moka
"""

FINANCIAL_TIPS = """
# Personal Finance Tips — Smart Finance Assistant

## Budgeting Basics
The 50/30/20 rule: 50% of income covers needs (rent, food, transport),
30% covers wants (gaming, cinema, gym, cafes), and 20% goes to savings.
Tracking every transaction including small tea purchases reveals
patterns that feel invisible day to day.

## Gaming Spending
Xbox Game Pass Ultimate is cost-effective if you play 3+ games per month.
Buying individual titles at full price often costs more than a 3-month pass.
Avoid impulse-buying DLC and in-game packs. Add them to a wishlist and
wait 48 hours before purchasing. Xbox Sales happen monthly and save
20 to 60 percent on most titles. Avoid buying V-Bucks or credits packs
unless you have a specific item in mind.

## Gym Membership
Monthly gym fees add up to $720 or more per year. Ask about off-peak
memberships which can be 20 to 30 percent cheaper if you train before 5pm.
Consolidate personal training sessions into a block booking as most gyms
offer a discount for purchasing 5 or 10 sessions upfront.

## Cinema at MCine
MCine loyalty cards and student discounts reduce ticket prices significantly.
Going on Tuesday or Wednesday saves $2 to $3 per ticket compared to weekends.
Pre-booking online avoids booking fees at the counter. Sharing snacks
instead of buying separately saves $5 to $8 per visit.

## Tea at Maison de Thé Moka
A daily visit to Maison de Thé costs $7.50 to $9.00 per visit. At three
visits per week that is over $100 per month. Purchasing quality loose-leaf
tea at home costs around $0.50 per cup, saving $7 or more per cup. Treat
Maison de Thé as a social or special occasion rather than a daily habit.

## Subscriptions Review
List all active subscriptions monthly including Game Pass, streaming
services, cloud storage, and fitness apps. Cancel any not used in the
last 30 days. The average person pays for 2 to 3 subscriptions they have
forgotten about entirely.

## Emergency Fund
Keep 3 months of living expenses in a separate high-interest savings
account. This prevents having to cancel gym memberships or skip
entertainment when an unexpected expense arises.

## Savings Goals
Set a specific goal such as saving $800 for a new Xbox controller and
headset by a certain date. Divide the goal by the number of weeks remaining
and automate that transfer each payday. Seeing the goal amount grow is more
motivating than a vague intention to spend less.

## Refunds and Impulse Purchases
Frequent refunds on gaming purchases are a sign of impulse buying.
A good rule: add any purchase over $20 to a wishlist and wait 48 hours.
If you still want it after 48 hours, buy it. This alone reduces impulse
game purchases by 30 to 40 percent.
"""

print("✅ Sample data ready!")
print(f"   CSV rows      : {SAMPLE_CSV.strip().count(chr(10))}")
print(f"   Knowledge base: {len(FINANCIAL_TIPS)} characters")

In [ ]:
def load_and_clean(file_or_string):
    """
    Load CSV data and clean the Amount column.
    Accepts: Gradio file object, raw CSV string, or file path.
    Returns: cleaned pandas DataFrame.
    """
    # Load from whatever source was provided
    try:
        if hasattr(file_or_string, 'name'):
            df = pd.read_csv(file_or_string.name)
        elif isinstance(file_or_string, str) and '\n' in file_or_string:
            df = pd.read_csv(io.StringIO(file_or_string))
        else:
            df = pd.read_csv(file_or_string)
    except Exception as e:
        raise ValueError(f"Could not read file: {e}")

    # Check required columns exist
    required = ['Date', 'Amount', 'Category', 'Description']
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Required: {required}")

    # Clean Amount — remove $, commas, spaces then convert to float
    df['Amount_Clean'] = pd.to_numeric(
        df['Amount'].astype(str).str.replace(r'[\$,\s]', '', regex=True),
        errors='coerce'
    )

    # Drop rows where amount could not be converted
    dropped = df['Amount_Clean'].isna().sum()
    if dropped > 0:
        print(f"⚠️  Dropped {dropped} row(s) with unreadable amounts.")
    df = df.dropna(subset=['Amount_Clean'])
    df['Category']    = df['Category'].fillna('Uncategorised')
    df['Description'] = df['Description'].fillna('')

    print(f"✅ Loaded {len(df)} transactions successfully.")
    return df


# --- Test immediately ---
df = load_and_clean(SAMPLE_CSV)
print(df[['Date','Amount','Amount_Clean','Category']].head(6))

In [ ]:
def analyse_spending(df):
    """
    Calculate spending totals, averages, and percentages by category.
    Separates refunds (negative amounts) from regular spending.
    Returns: dict with summary DataFrame and key totals.
    """
    # Separate positive spending from refunds
    positive = df[df['Amount_Clean'] > 0].copy()
    refunds  = df[df['Amount_Clean'] < 0].copy()

    total_spent   = positive['Amount_Clean'].sum()
    total_refunds = abs(refunds['Amount_Clean'].sum())
    net_spent     = total_spent - total_refunds

    # Group by category
    summary = (
        positive
        .groupby('Category')['Amount_Clean']
        .agg(Total_Spent='sum', Avg_Transaction='mean', Count='count')
        .round(2)
        .reset_index()
    )

    # Add percentage of total spending
    summary['Percentage'] = (summary['Total_Spent'] / total_spent * 100).round(1)
    summary = summary.sort_values('Total_Spent', ascending=False).reset_index(drop=True)

    return {
        'summary': summary,
        'total':   round(total_spent, 2),
        'refunds': round(total_refunds, 2),
        'net':     round(net_spent, 2),
        'avg':     round(positive['Amount_Clean'].mean(), 2),
        'n':       len(positive),
        'top':     summary.iloc[0]['Category']
    }


# --- Test immediately ---
a = analyse_spending(df)
print("📊 Category Summary:")
print(a['summary'].to_string(index=False))
print(f"\nTotal: ${a['total']}  |  Refunds: ${a['refunds']}  |  Net: ${a['net']}")

In [ ]:
def make_recommendations(a):
    """
    Generate personalised spending tips based on category totals.
    Triggers specific advice for Gaming, Gym, Cinema, Tea & Café.
    Returns: formatted string of recommendations.
    """
    tips = []

    for _, row in a['summary'].iterrows():
        cat, spent, pct = row['Category'], row['Total_Spent'], row['Percentage']

        if cat == 'Gaming' and spent > 50:
            saving = round(spent * 0.4, 2)
            tips.append(
                f"🎮 Gaming: You spent ${spent:.2f} on Xbox. Using the 48-hour "
                f"wishlist rule before every purchase could save ~${saving:.2f}/month "
                f"on impulse buys."
            )
        if cat == 'Gym' and spent > 80:
            tips.append(
                f"💪 Gym: ${spent:.2f} this period. Ask about off-peak membership "
                f"rates or block-book personal training sessions for a 10–20% discount."
            )
        if cat == 'Cinema' and spent > 40:
            tips.append(
                f"🎬 Cinema (MCine): ${spent:.2f} spent. Going on Tuesday or Wednesday "
                f"saves $2–3 per ticket. A MCine loyalty card reduces costs further."
            )
        if cat == 'Tea & Café' and spent > 30:
            brew_save = round(spent * 0.85, 2)
            tips.append(
                f"🍵 Tea & Café (Maison de Thé): ${spent:.2f} — brewing quality "
                f"loose-leaf tea at home could save ~${brew_save:.2f}/month. "
                f"Save visits for social occasions."
            )

    if not tips:
        tips.append("✅ Spending looks balanced! Keep tracking to maintain good habits.")

    tips.append(
        f"\n💡 Biggest spend: {a['top']} — "
        f"focus here first to make the biggest savings impact."
    )
    return "\n".join(tips)


# --- Test immediately ---
print("💡 Recommendations:")
print(make_recommendations(a))

In [ ]:
def full_report(a):
    """
    Combine all analysis into one readable text report.
    Includes a text-art bar chart showing spending visually.
    Returns: formatted string ready to display in Gradio.
    """
    lines = [
        "=" * 56,
        "        SMART FINANCE ASSISTANT REPORT",
        "=" * 56,
        f"  Transactions    : {a['n']}",
        f"  Total Spent     : ${a['total']:>9.2f}",
        f"  Total Refunds   : ${a['refunds']:>9.2f}",
        f"  Net Spent       : ${a['net']:>9.2f}",
        f"  Avg Transaction : ${a['avg']:>9.2f}",
        "",
        "── Spending by Category ────────────────────────────",
    ]

    for _, row in a['summary'].iterrows():
        bar = "█" * max(1, int(row['Percentage'] / 5))
        lines.append(
            f"  {row['Category']:<15} ${row['Total_Spent']:>7.2f} "
            f"({row['Percentage']:>5.1f}%)  {bar}"
        )

    lines += [
        "",
        "── Recommendations ─────────────────────────────────",
        make_recommendations(a),
        "=" * 56,
    ]
    return "\n".join(lines)


# --- Test immediately ---
print(full_report(a))

In [ ]:
def savings_calculator(income, expenses, goal):
    """
    Calculate how long it takes to reach a savings goal.
    Validates all inputs and handles edge cases gracefully.
    Args: income, expenses, goal — monthly amounts in dollars.
    Returns: formatted results string.
    """
    income, expenses, goal = float(income), float(expenses), float(goal)

    if income <= 0:
        return "❌ Monthly income must be greater than zero."
    if expenses < 0:
        return "❌ Monthly expenses cannot be negative."
    if goal <= 0:
        return "❌ Savings goal must be greater than zero."

    monthly_savings = income - expenses
    if monthly_savings <= 0:
        shortfall = abs(monthly_savings)
        return (f"❌ You are spending ${shortfall:.2f} more than you earn each month.\n"
                f"   Reduce expenses by at least ${shortfall + 1:.2f} to start saving.")

    months_total = goal / monthly_savings
    years        = int(months_total // 12)
    months_rem   = int(months_total % 12) + 1
    weekly       = monthly_savings / 4.33
    daily        = monthly_savings / 30
    rate         = monthly_savings / income * 100

    lines = [
        "=" * 51,
        "          SAVINGS GOAL CALCULATOR",
        "=" * 51,
        f"  Monthly Income    : ${income:>10.2f}",
        f"  Monthly Expenses  : ${expenses:>10.2f}",
        f"  Monthly Savings   : ${monthly_savings:>10.2f}  ({rate:.1f}%)",
        f"  Savings Goal      : ${goal:>10.2f}",
        "",
        "── Time to Reach Goal ──────────────────────────────",
        (f"  ⏱️   {years} year(s) and {months_rem} month(s)" if years
         else f"  ⏱️   {months_rem} month(s)"),
        "",
        "── Breakdown ───────────────────────────────────────",
        f"  Per Week : ${weekly:.2f}",
        f"  Per Day  : ${daily:.2f}",
        "",
        "── Verdict ─────────────────────────────────────────",
        (f"  ✅ {rate:.1f}% savings rate — above the recommended 20%!"
         if rate >= 20 else
         f"  💡 {rate:.1f}% savings rate — aim for 20% by cutting "
         f"gaming impulse buys and daily café visits."),
        "=" * 51,
    ]
    return "\n".join(lines)


# Register as agent tool
try:
    from hands_on_ai import agent
    agent.register_tool('savings_calculator', savings_calculator)
    print("✅ Registered savings_calculator as agent tool.")
except:
    print("ℹ️  Running savings_calculator as standalone function.")

# --- Test immediately ---
print(savings_calculator(2500, 1800, 5000))

In [ ]:
import urllib.request
import json

SYSTEM_PROMPT = """You are FinBot, a friendly personal finance advisor for a young person
who enjoys Xbox gaming, going to the gym, watching movies at MCine in Moka,
and drinking tea at Maison de Thé in Moka.
Help them understand their spending, budget better, and build savings habits
around their lifestyle. Be concise, practical, and encouraging.
Use dollar amounts. Never give stock investment advice.
If spending context is provided, reference it to give personalised advice."""

chat_history = []

def call_anthropic(user_msg, system):
    """Call Anthropic API directly as fallback."""
    url  = "https://api.anthropic.com/v1/messages"
    body = json.dumps({
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 512,
        "system": system,
        "messages": [{"role": "user", "content": user_msg}]
    }).encode()

    req = urllib.request.Request(url, data=body, headers={
        "Content-Type":      "application/json",
        "anthropic-version": "2023-06-01"
    })

    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["content"][0]["text"]
    except Exception as e:
        return f"❌ Error: {e}"


def chat_finbot(user_msg, spending_ctx=""):
    """
    Send a message to FinBot and return its reply.
    Tries hands-on-ai first, falls back to Anthropic API if unavailable.
    spending_ctx: optional spending summary for personalised advice.
    """
    global chat_history
    system = SYSTEM_PROMPT
    if spending_ctx:
        system += f"\n\n[User's current spending: {spending_ctx}]"

    # Try hands-on-ai first
    try:
        reply = get_response(user_msg, system=system)
        if reply and "Connection error" not in reply:
            chat_history.append((user_msg, reply))
            return reply
    except Exception:
        pass

    # Fallback: Anthropic API
    print("ℹ️  hands-on-ai unavailable — using Anthropic API fallback.")
    reply = call_anthropic(user_msg, system)
    chat_history.append((user_msg, reply))
    return reply


# --- Test immediately ---
reply = chat_finbot(
    "I spent $216 on Xbox games this month. Is that too much?",
    spending_ctx="Gaming $216.96, Gym $85.00, Cinema $35.00, Tea & Café $32.50"
)
print("FinBot says:")
print(reply)

In [ ]:
def rag_lookup(question):
    """
    Answer a finance question using the tips knowledge base.
    Tries hands_on_ai RAG first, falls back to Anthropic API if unavailable.
    """
    # Try hands-on-ai RAG first
    try:
        from hands_on_ai.rag import get_answer
        answer = get_answer(question, documents=[FINANCIAL_TIPS])
        if answer and "Connection error" not in answer:
            return answer
    except Exception:
        pass

    # Fallback: send document + question directly to Anthropic API
    print("ℹ️  hands-on-ai RAG unavailable — using Anthropic API fallback.")
    system = ("You are a financial knowledge assistant. "
              "Answer questions using ONLY the document provided. "
              "If the answer is not in the document, say so clearly.")
    prompt = (f"DOCUMENT:\n{FINANCIAL_TIPS}\n\n"
              f"QUESTION: {question}\n\n"
              f"Answer using only the document above:")
    return call_anthropic(prompt, system)


# --- Test immediately ---
print("RAG Answer:")
print(rag_lookup("How can I spend less money on tea at the café?"))

In [ ]:
import gradio as gr

# Wrapper: Spending Analysis tab
def ui_analyse(file):
    try:
        src  = file if file else SAMPLE_CSV
        note = "" if file else "[Using sample data — upload your CSV above]\n\n"
        df2  = load_and_clean(src)
        return note + full_report(analyse_spending(df2))
    except ValueError as e:
        return (f"❌ {e}\n\n"
                f"Your CSV needs these columns:\n"
                f"  Date, Amount, Category, Description")
    except Exception as e:
        return f"❌ Unexpected error: {e}"


# Wrapper: Chat tab
def ui_chat(msg, history, file):
    if not msg.strip():
        return history, ""
    ctx = ""
    if file:
        try:
            df2  = load_and_clean(file)
            a2   = analyse_spending(df2)
            top4 = a2['summary'].head(4)
            ctx  = (f"Net spent ${a2['net']}, top categories: " +
                    ", ".join(f"{r.Category} ${r.Total_Spent}"
                              for r in top4.itertuples()))
        except:
            pass
    reply   = chat_finbot(msg, ctx)
    history = (history or []) + [(msg, reply)]
    return history, ""


# Wrapper: RAG tab
def ui_rag(q):
    return rag_lookup(q) if q.strip() else "Please type a question first."


# Wrapper: Savings tab
def ui_savings(inc, exp, goal):
    try:
        return savings_calculator(float(inc), float(exp), float(goal))
    except:
        return "❌ Please enter valid numbers in all three fields."


# Build the Gradio app
with gr.Blocks(title="Smart Finance Assistant", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 💰 Smart Finance Assistant\n"
                "*ISYS2001 — Track your gaming, gym, cinema & café spending*")

    with gr.Tabs():

        with gr.Tab("📊 Spending Analysis"):
            gr.Markdown("Upload your CSV or leave blank to use the March–May 2026 sample data.")
            f1   = gr.File(label="Upload CSV", file_types=[".csv"])
            btn1 = gr.Button("Analyse My Spending", variant="primary")
            out1 = gr.Textbox(label="Report", lines=30)
            btn1.click(ui_analyse, inputs=f1, outputs=out1)

        with gr.Tab("💬 Chat Advisor"):
            gr.Markdown("Ask FinBot anything. Upload your CSV for personalised advice.")
            f2   = gr.File(label="Upload CSV (optional)", file_types=[".csv"])
            bot  = gr.Chatbot(label="FinBot", height=380)
            msg  = gr.Textbox(label="Your message",
                              placeholder="e.g. Am I spending too much on Xbox games?",
                              lines=2)
            btn2 = gr.Button("Send", variant="primary")
            btn2.click(ui_chat, [msg, bot, f2], [bot, msg])
            msg.submit(ui_chat,  [msg, bot, f2], [bot, msg])

        with gr.Tab("📚 Finance Tips (RAG)"):
            gr.Markdown("Ask a finance question — answered from our knowledge base.")
            q3   = gr.Textbox(label="Your Question",
                              placeholder="e.g. How do I stop impulse buying Xbox games?")
            btn3 = gr.Button("Get Advice", variant="primary")
            out3 = gr.Textbox(label="Answer", lines=12)
            btn3.click(ui_rag, inputs=q3, outputs=out3)

        with gr.Tab("🎯 Savings Calculator"):
            gr.Markdown("Calculate how long until you reach your savings goal.")
            with gr.Row():
                inc  = gr.Number(label="Monthly Income ($)",   value=2500)
                exp  = gr.Number(label="Monthly Expenses ($)", value=1800)
                goal = gr.Number(label="Savings Goal ($)",     value=5000)
            btn4 = gr.Button("Calculate", variant="primary")
            out4 = gr.Textbox(label="Your Savings Plan", lines=16)
            btn4.click(ui_savings, [inc, exp, goal], out4)

    gr.Markdown("---\n*Smart Finance Assistant | ISYS2001 | Python + hands-on-ai + Gradio*")

demo.launch(share=True)